In [3]:
#Just trail on embedding layer in RNN, this is for converting the words into sentences before putting it inside the RNN
from tensorflow.keras.preprocessing.text import one_hot

In [1]:
!pip install tensorflow

In [2]:
!pip install scikeras

In [6]:
### sentences
sent=[  'the glass of milk',
     'the glass of juice',
     'the cup of tea',
    'I am a good boy',
     'I am a good developer',
     'understand the meaning of words',
     'your videos are good',]

vocab_size = 10000

In [7]:
one_hot_encoded = [one_hot(word,vocab_size) for word in sent]

In [8]:
one_hot_encoded

[[6905, 6685, 1238, 4229],
 [6905, 6685, 1238, 8416],
 [6905, 9599, 1238, 9523],
 [6935, 6146, 8250, 7788, 925],
 [6935, 6146, 8250, 7788, 4572],
 [6608, 6905, 9458, 1238, 1647],
 [1727, 8039, 8368, 7788]]

In [11]:
## Here the vocab_size is the number of words in the whole document
## Now we also need to make sure all the array length is the same for everything, so
from tensorflow.keras.utils import pad_sequences
one_samelength = pad_sequences(one_hot_encoded,maxlen=8,padding='pre')

In [12]:
one_samelength

array([[   0,    0,    0,    0, 6905, 6685, 1238, 4229],
       [   0,    0,    0,    0, 6905, 6685, 1238, 8416],
       [   0,    0,    0,    0, 6905, 9599, 1238, 9523],
       [   0,    0,    0, 6935, 6146, 8250, 7788,  925],
       [   0,    0,    0, 6935, 6146, 8250, 7788, 4572],
       [   0,    0,    0, 6608, 6905, 9458, 1238, 1647],
       [   0,    0,    0,    0, 1727, 8039, 8368, 7788]], dtype=int32)

In [16]:
## Now building the embedding layer(Which internally uses word2vec model and we can fix the number of dimensions(features), as these are basically feature representations)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
model = Sequential()
model.add(Embedding(vocab_size,output_dim=10,input_length=8))
## So the above line does not have any input it is just randomly intialized numbers and filled the matrix with so and so shape
model.compile('adam','mse')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [17]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [18]:
one_samelength[0]

array([   0,    0,    0,    0, 6905, 6685, 1238, 4229], dtype=int32)

In [21]:
model.predict(one_samelength[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step


array([[-0.04240158, -0.02828951,  0.0159237 ,  0.04143759,  0.01083356,
         0.01779257,  0.03788904, -0.02238923, -0.00422553,  0.03922004],
       [-0.04240158, -0.02828951,  0.0159237 ,  0.04143759,  0.01083356,
         0.01779257,  0.03788904, -0.02238923, -0.00422553,  0.03922004],
       [-0.04240158, -0.02828951,  0.0159237 ,  0.04143759,  0.01083356,
         0.01779257,  0.03788904, -0.02238923, -0.00422553,  0.03922004],
       [-0.04240158, -0.02828951,  0.0159237 ,  0.04143759,  0.01083356,
         0.01779257,  0.03788904, -0.02238923, -0.00422553,  0.03922004],
       [-0.03501258, -0.03511455, -0.02398294,  0.03959315, -0.01419338,
         0.02505944,  0.00964885, -0.02866111,  0.03372327,  0.04149849],
       [-0.00895495, -0.0069621 ,  0.00212801,  0.03061003, -0.04999164,
        -0.03664589, -0.04603106, -0.0350795 ,  0.02875134,  0.04538519],
       [-0.03567605, -0.04993637,  0.02621548,  0.00791319, -0.02262059,
         0.03074041, -0.00529804,  0.01323918

So now you have a 10-dimensional feature representation for the 1st sentence which is present in the one_samelength

## RNN

In [23]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [24]:
max_features = 10000 #Vocabulary size
(X_train,y_train),(X_test,y_test) = imdb.load_data(num_words=max_features)
X_train.shape

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


(25000,)

In [26]:
# The data is already onehot encoded for us, so we gotta see if we can match it back
names = imdb.get_word_index()
dict = {key : value for key, value in names.items() }

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [27]:
dict

{'fawn': 34701,
 'tsukino': 52006,
 'nunnery': 52007,
 'sonja': 16816,
 'vani': 63951,
 'woods': 1408,
 'spiders': 16115,
 'hanging': 2345,
 'woody': 2289,
 'trawling': 52008,
 "hold's": 52009,
 'comically': 11307,
 'localized': 40830,
 'disobeying': 30568,
 "'royale": 52010,
 "harpo's": 40831,
 'canet': 52011,
 'aileen': 19313,
 'acurately': 52012,
 "diplomat's": 52013,
 'rickman': 25242,
 'arranged': 6746,
 'rumbustious': 52014,
 'familiarness': 52015,
 "spider'": 52016,
 'hahahah': 68804,
 "wood'": 52017,
 'transvestism': 40833,
 "hangin'": 34702,
 'bringing': 2338,
 'seamier': 40834,
 'wooded': 34703,
 'bravora': 52018,
 'grueling': 16817,
 'wooden': 1636,
 'wednesday': 16818,
 "'prix": 52019,
 'altagracia': 34704,
 'circuitry': 52020,
 'crotch': 11585,
 'busybody': 57766,
 "tart'n'tangy": 52021,
 'burgade': 14129,
 'thrace': 52023,
 "tom's": 11038,
 'snuggles': 52025,
 'francesco': 29114,
 'complainers': 52027,
 'templarios': 52125,
 '272': 40835,
 '273': 52028,
 'zaniacs': 52130,

In [29]:
# PREPROCESSING making all the sentence length the same
max_len = 500 # the maximum length of a sentence will be only 500
X_train = sequence.pad_sequences(X_train,maxlen=max_len)
X_test = sequence.pad_sequences(X_test,maxlen=max_len)

In [40]:
#Now building the model
model = Sequential()
model.add(Embedding(input_dim=max_features,output_dim=128,input_length=max_len)) #input_length = the sentence length (depreceated), input_dim = size of vocabulary, output_dim will be how many features u need for a word (like 300 in word2vec)
model.add(SimpleRNN(128,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [44]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Input

model = Sequential([
    Input(shape=(max_len,)),
    Embedding(input_dim=max_features, output_dim=128),
    SimpleRNN(128, activation='tanh'), #RNN uses tanh it seems
    Dense(1, activation='sigmoid')
])

[embedding documentation](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding)

In [49]:
model.compile(metrics=['accuracy'],optimizer='adam',loss='binary_crossentropy')

In [45]:
model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,025 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

In [47]:
from tensorflow.keras.callbacks import EarlyStopping
earlystopping = EarlyStopping(monitor = 'val_loss',patience =5, restore_best_weights= True)
earlystopping

In [50]:
# TRAINING THE MODEL
history = model.fit(
    X_train,y_train, epochs = 10, batch_size= 32,
    validation_split = 0.2,
    callbacks = [earlystopping]

)


Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 151s 238ms/step - accuracy: 0.6618 - loss: 25788.9316 - val_accuracy: 0.7918 - val_loss: 0.4419
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 199s 234ms/step - accuracy: 0.8337 - loss: 0.3871 - val_accuracy: 0.7960 - val_loss: 0.4362
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 145s 232ms/step - accuracy: 0.8862 - loss: 0.2746 - val_accuracy: 0.8060 - val_loss: 0.4366
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 145s 232ms/step - accuracy: 0.9136 - loss: 0.2148 - val_accuracy: 0.8118 - val_loss: 0.4602
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 204s 236ms/step - accuracy: 0.9291 - loss: 0.1858 - val_accuracy: 0.8084 - val_loss: 0.4752
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 200s 233ms/step - accuracy: 0.9368 - loss: 0.1723 - val_accuracy: 0.8018 - val_loss: 0.5535
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 201s 231ms/step - accuracy: 0.9535 - loss: 0.1281 - val_accuracy: 0.8134 - val_loss: 0.5272


In [53]:
# model.save('C://Users//Nandhika//Documents//RNN_Imdb//models//SimpleRNN_model_imdb.h5')
model.save("/content/sample_data/RNN_model_imdb.h5")